# A1: Predicting Car Price

AT82.03 Machine Learning

## 1. Loading the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/Cars.csv")
print(df.shape)   # raw data: expect (8128, 13)
df.head()

## 2. Data cleaning

In [ ]:
# Start from the raw file every time, so this cell is safe to run again
df = pd.read_csv("data/Cars.csv")

# Owner: turn the text categories into numbers 1-5, as the brief asks
df["owner"] = df["owner"].map({
    "First Owner": 1, "Second Owner": 2, "Third Owner": 3,
    "Fourth & Above Owner": 4, "Test Drive Car": 5,
})

# Fuel: drop CNG and LPG, because their mileage is in km/kg, not kmpl
df = df[~df["fuel"].isin(["CNG", "LPG"])]

# Test drive cars are unrealistically expensive, so remove them
df = df[df["owner"] != 5]

# Brand: keep only the first word of the car name ("Maruti Swift Dzire" -> "Maruti")
df["brand"] = df["name"].str.split().str[0]
df = df.drop(columns=["name"])

# Mileage, engine, max_power: remove the units ("23.4 kmpl" -> 23.4) and convert to numbers.
# errors="coerce" turns broken entries (e.g. "bhp" with no number) into missing values
for col in ["mileage", "engine", "max_power"]:
    df[col] = pd.to_numeric(df[col].str.split().str[0], errors="coerce")

# Torque: dropped, as the brief instructs
df = df.drop(columns=["torque"])

# Put brand first so the table matches Figure 1 in the brief
df = df[["brand"] + [c for c in df.columns if c != "brand"]]

# Check the result
print(df.shape)              # expect (8028, 12)
print(df.isna().sum())       # missing values per column
print("Duplicates:", df.duplicated().sum())
df.head()

## 3. Exploratory Data Analysis (EDA)

Before modelling, look at the data: how prices are spread, which features move with price, and anything odd.

### 3.1 Summary statistics and duplicates

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic statistics for every numeric column (count, mean, min, max, quartiles)
display(df.describe().round(2).T)

# The dataset has no ID column, so identical rows are probably the same listing posted twice.
# Keeping them would let the same car appear in both the training and test sets (leakage),
# which makes test scores look better than they really are, so drop them.
print("Rows before dropping duplicates:", len(df))
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after dropping duplicates: ", len(df))

### 3.2 How is the selling price distributed?

In [ ]:
# Left: raw price. Right: log of price (what the brief asks us to predict).
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["selling_price"], bins=50, ax=axes[0])
axes[0].set_title("Selling price")
sns.histplot(np.log(df["selling_price"]), bins=50, ax=axes[1])
axes[1].set_title("log(selling price)")
plt.tight_layout()
plt.show()

### 3.3 Which numeric features move with price?

In [ ]:
# Correlation of every numeric feature with log(price), shown as a heatmap.
num = df.select_dtypes("number").copy()
num["log_price"] = np.log(num["selling_price"])
num = num.drop(columns=["selling_price"])

plt.figure(figsize=(9, 7))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between numeric features")
plt.show()

# The same numbers as a ranked list, so it is easy to read
print(num.corr()["log_price"].drop("log_price").sort_values(ascending=False).round(2))

### 3.4 The strongest numeric features against price

In [ ]:
# Scatter plots of the three features most correlated with log(price)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["max_power", "year", "engine"]):
    sns.scatterplot(x=df[col], y=np.log(df["selling_price"]), alpha=0.3, ax=ax)
    ax.set_ylabel("log(selling price)")
    ax.set_title(f"{col} vs log(price)")
plt.tight_layout()
plt.show()

### 3.5 Categorical features against price

In [ ]:
# Box plots: the middle line is the median price for each group
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["fuel", "transmission", "seller_type", "owner"]):
    sns.boxplot(x=df[col], y=np.log(df["selling_price"]), ax=ax)
    ax.set_ylabel("log(selling price)")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# Brands: how many cars each brand has (top 15)
plt.figure(figsize=(12, 4))
df["brand"].value_counts().head(15).plot(kind="bar")
plt.title("Number of cars per brand (top 15)")
plt.ylabel("Count")
plt.show()

### 3.6 What the EDA shows

*(Write this in your own words after looking at your plots. Points to cover:)*

- **Price is heavily skewed**: most cars are cheap and a few are very expensive. Taking the log makes the distribution much more balanced, which is why we predict log(price).
- **Strongest numeric features**: year (about 0.72) and max_power (about 0.65), then engine (about 0.47). Newer, more powerful cars cost more.
- **Weak features**: mileage has almost no correlation with price (about 0.02); seats is weak (about 0.27).
- **Negative features**: owner (about -0.39) and km_driven (about -0.19): cars that have had more owners or been driven more sell for less.
- **Categorical**: automatic cars, diesel cars and dealer listings have higher median prices than manual, petrol and individual listings.
- **Correlated inputs**: engine and max_power are strongly correlated with each other, so they carry overlapping information.